# ASR-course code to load in the models for finetuning

This notebook is a replication of the code used by the Whisper Finetuning event. Available here: https://github.com/huggingface/community-events/tree/main/whisper-fine-tuning-event.
Our approach is based on the paper by Jain et al. [1] Available here: https://doi.org/10.1109/ACCESS.2024.3378738
It is only slightly adapted to work with our hardware and datasets.
All comments in this notebook are there to explain the highlight the changes we made to the original notebook.

In [ ]:
from pathlib import Path
import random
from sklearn.model_selection import train_test_split

Here we load in the datasets, as closely as possible to how it was done in the source paper. 

In [ ]:
CMU_data = Path("cmukids_clean")

CMU_audio = sorted(CMU_data.glob("*.wav"))

CMU_samples = []

random.seed(42)

for wav_path in CMU_audio:
    txt_path = wav_path.with_suffix(".txt")
    
    if txt_path.exists():
        with open(txt_path, "r", encoding="utf-8") as f:
            text = f.read().strip()

        CMU_samples.append({
            "audio_path": str(wav_path),
            "text": text
        })
random.shuffle(CMU_samples)

split = int(len(CMU_samples) * 0.8)

CMU_train = CMU_samples[:split]
CMU_test = CMU_samples[split:]

CMU_train, CMU_val = train_test_split(
    CMU_train,
    test_size=0.1,
    random_state=42,
    shuffle=True
)

print(len(CMU_samples))
print(len(CMU_train))
print(len(CMU_val))
print(len(CMU_test))

In [ ]:
dataset_path = Path("Myst_train")

audio_files = sorted(dataset_path.glob("*.wav"))

Myst_data = []

for wav_path in audio_files:
    txt_path = wav_path.with_suffix(".txt")
    
    if txt_path.exists():
        with open(txt_path, "r", encoding="utf-8") as f:
            text = f.read().strip()

        Myst_data.append({
            "audio_path": str(wav_path),
            "text": text
        })

Myst_train, Myst_val = train_test_split(
    Myst_data,
    test_size=0.1,
    random_state=42,
    shuffle=True
)

print(len(Myst_data))
print(len(Myst_train))
print(len(Myst_val))

In [ ]:
Myst_test = Path("Myst_test")

test_audio = sorted(Myst_test.glob("*.wav"))

Myst_test = []

for wav_path in test_audio:
    txt_path = wav_path.with_suffix(".txt")
    
    if txt_path.exists():
        with open(txt_path, "r", encoding="utf-8") as f:
            text = f.read().strip()

        Myst_test.append({
            "audio_path": str(wav_path),
            "text": text
        })
print(len(Myst_test))

In [ ]:
dataset_path = Path("TLT2021_train")

audio_files = sorted(dataset_path.glob("*.wav"))
TLT_samples = []

for wav_path in audio_files:
    txt_path = wav_path.with_suffix(".txt")
    
    if txt_path.exists():
        with open(txt_path, "r", encoding="utf-8") as f:
            text = f.read().strip().lower()

        TLT_samples.append({
            "id": wav_path.stem,
            "audio_path": str(wav_path),
            "text": text
        })

TLT_train, TLT_val = train_test_split(
    TLT_samples,
    test_size=0.1,
    random_state=42,
    shuffle=True
)

len(TLT_train), len(TLT_val)

TLT_test is split, because otherwise it was too big for ponyland to load in at once.

In [ ]:
dataset_path = Path("TLT2021_test1")

audio_files = sorted(dataset_path.glob("*.wav"))
TLT_test1 = []

for wav_path in audio_files:
    txt_path = wav_path.with_suffix(".txt")
    
    if txt_path.exists():
        with open(txt_path, "r", encoding="utf-8") as f:
            text = f.read().strip().lower()

        TLT_test1.append({
            "audio_path": str(wav_path),
            "text": text
        })

len(TLT_test1)

In [ ]:
dataset_path = Path("TLT2021_test2")

audio_files = sorted(dataset_path.glob("*.wav"))
TLT_test2 = []

for wav_path in audio_files:
    txt_path = wav_path.with_suffix(".txt")
    
    if txt_path.exists():
        with open(txt_path, "r", encoding="utf-8") as f:
            text = f.read().strip().lower()

        TLT_test2.append({
            "audio_path": str(wav_path),
            "text": text
        })

len(TLT_test2)
TLT_test = TLT_test1 + TLT_test2

In [ ]:
len(TLT_test)

Code to load in TLT2021 with speed augmentation AFTER validation set Ids are removed from the file.

In [ ]:

dataset_path = Path("TLT2021_speed1")

audio_files = sorted(dataset_path.glob("*.wav"))

TLT_speed1 = []

for wav_path in audio_files:
    txt_path = wav_path.with_suffix(".txt")
    
    if txt_path.exists():
        with open(txt_path, "r", encoding="utf-8") as f:
            text = f.read().strip().lower()

        TLT_speed1.append({
            "audio_path": str(wav_path),
            "text": text
        })

print(len(TLT_speed1))

In [ ]:
TLT_speed_data1 = Path("TLT2021_speed2")

audio = sorted(TLT_speed_data1.glob("*.wav"))

TLT_speed2 = []


for wav_path in audio_files:
    txt_path = wav_path.with_suffix(".txt")
    
    if txt_path.exists():
        with open(txt_path, "r", encoding="utf-8") as f:
            text = f.read().strip().lower()

        TLT_speed2.append({
            "audio_path": str(wav_path),
            "text": text
        })

print(len(TLT_speed2))

In [ ]:
TLT_speed = TLT_speed1 + TLT_speed2
len(TLT_speed)

Combine the data into train and validation sets. Use TLT_speed instead of TLT_train when loading speed augmentation train set

In [ ]:
train_data = Myst_train + CMU_train + TLT_train # + TLT_speed
val_data = CMU_val + Myst_val + TLT_val

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_list(train_data)
Myst_test_dataset = Dataset.from_list(Myst_test)
CMU_test_dataset = Dataset.from_list(CMU_test)
TLT_test_dataset = Dataset.from_list(TLT_test)
val_dataset = Dataset.from_list(val_data)
train_dataset

In [ ]:
from transformers import WhisperFeatureExtractor

feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-small")

In [ ]:
from transformers import WhisperTokenizer

tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-small")

In [ ]:
from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained("openai/whisper-small")

In [ ]:
from transformers.models.whisper.english_normalizer import BasicTextNormalizer

do_lower_case = False
do_remove_punctuation = False

normalizer = BasicTextNormalizer()

In [ ]:
import librosa

def prepare_dataset(batch):

    audio, sr = librosa.load(batch["audio_path"], sr=16000)
    batch["input_features"] = processor.feature_extractor(
        audio,
        sampling_rate=16000
    ).input_features[0]
    batch["labels"] = processor.tokenizer(batch["text"]).input_ids
    batch["input_length"] = len(audio) / sr
    return batch

In [ ]:
train_dataset = train_dataset.map(prepare_dataset, remove_columns=["audio_path", "text"])
Myst_test_dataset = Myst_test_dataset.map(prepare_dataset, remove_columns=["audio_path", "text"])
CMU_test_dataset = CMU_test_dataset.map(prepare_dataset, remove_columns=["audio_path", "text"])
TLT_test_dataset = TLT_test_dataset.map(prepare_dataset, remove_columns=["audio_path", "text"])
val_dataset = val_dataset.map(prepare_dataset, remove_columns=["audio_path", "text"])

In [ ]:
max_input_length = 30.0

def is_audio_in_length_range(length):
    return length < max_input_length

In [ ]:
train_dataset = train_dataset.filter(is_audio_in_length_range, input_columns=["input_length"])
Myst_test_dataset = Myst_test_dataset.filter(is_audio_in_length_range, input_columns=["input_length"])
CMU_test_dataset = CMU_test_dataset.filter(is_audio_in_length_range, input_columns=["input_length"])
TLT_test_dataset = TLT_test_dataset.filter(is_audio_in_length_range, input_columns=["input_length"])
val_dataset = val_dataset.filter(is_audio_in_length_range, input_columns=["input_length"])

Saving the datasets to Disk, so they can be easily used in the finetuning notebook.

In [ ]:
train_dataset.save_to_disk("TLT_train_dataset")
Myst_test_dataset.save_to_disk("Myst_test_dataset")
CMU_test_dataset.save_to_disk("CMU_test_dataset")
TLT_test_dataset.save_to_disk("TLT_test_dataset")
val_dataset.save_to_disk("TLT_val_dataset_fix")

# References

[1] Jain, Rishabh & Barcovschi, Andrei & Yiwere, Mariam & Corcoran, Peter & Cucu, Horia. (2024). Exploring Native and Non-Native English Child Speech Recognition With Whisper. IEEE Access. PP. 1-1. 10.1109/ACCESS.2024.3378738. 